# Recitation 0.18: Weights and Biases (WandB)

In this recitation, you will learn about performance visualization, version controlling, model tracking, collboration strategies, hyperparameter tuning sweeps and many more using [WandB](https://wandb.ai/).

Author: Ron Sarma

## Installation and Libraries

In [1]:
!pip install wandb -qqq

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision.transforms import ToTensor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device: ", device)

from tqdm import tqdm

Device:  cuda


In [3]:
import wandb, os
# os.environ['WANDB_API_KEY'] = "" #your key here (starts with wandb_v...)
wandb.login() # It is a better practice to let wandb prompt for the API key securely, rather than hardcoding it.

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rsarma (rsarma-organization) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Helper functions and Model

### 1. Data Preparation
Before we dive into Weights & Biases, we need a dataset and a model. In the next few cells, we download the standard CIFAR-10 image dataset and create helper functions to load it in batches using PyTorch `DataLoader`s.

In [6]:
data_train = datasets.CIFAR10(
    root = 'data',
    train = True,
    transform = ToTensor(),
    download = True,
)
data_test = datasets.CIFAR10(
    root = 'data',
    train = False,
    download = True,
    transform = ToTensor()
)

100%|██████████| 170M/170M [27:31<00:00, 103kB/s] 


In [7]:
def build_data(batch_size, data_train, data_test):
    train_loader = torch.utils.data.DataLoader(data_train, batch_size=batch_size, shuffle=True)
    test_loader = torch.utils.data.DataLoader(data_test, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

### 2. Defining the Model
Next, we define a simple Convolutional Neural Network (CNN) to classify the images. This is the model architecture whose performance we will track.

In [8]:
class Network(nn.Module):

  def __init__(self):

    super(Network, self).__init__()

    self.CNN = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=2),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AvgPool2d(kernel_size=9),
            nn.Flatten()
    )

    self.classification = nn.Linear(576, 10)
  def forward(self, x):

    x_cnn = self.CNN(x)
    res = self.classification(x_cnn)

    return res

model = Network().to(device)
print(model)

Network(
  (CNN): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): AvgPool2d(kernel_size=9, stride=9, padding=0)
    (4): Flatten(start_dim=1, end_dim=-1)
  )
  (classification): Linear(in_features=576, out_features=10, bias=True)
)


In [9]:
train_loader, test_loader = build_data(64, data_train, data_test)

for x, y in train_loader:
  break
model(x.to(device)).shape

torch.Size([64, 10])

In [10]:
def get_optim(optimizer, learning_rate, model):
  if optimizer=='sgd':
    return optim.SGD(model.parameters(), lr=learning_rate)
  else:
    return optim.Adam(model.parameters(), lr=learning_rate)

### 3. The Training Loop & WandB Logging
Here is where the WandB integration happens!
- `train_epoch`: A standard PyTorch function to handle the math for one epoch.
- `train`: The main loop. Pay close attention to **`wandb.log(metrics)`**, which sends our training loss and accuracy to the WandB dashboard in real-time. We also use **`wandb.Artifact`** to securely save the model's weights (`Model.pth`) and link them to this specific run every time the accuracy improves.

In [11]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    num_correct = 0
    total_loss = 0

    for i, (x, y) in enumerate(loader):
          optimizer.zero_grad()

          x = x.cuda()
          y = y.cuda()

          # Updated to modern PyTorch AMP syntax
          with torch.amp.autocast('cuda'):
              outputs = model(x)
              loss = criterion(outputs, y)

          total_loss += float(loss)

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()
    ep_loss = float(total_loss / len(loader))

    return model, ep_loss

In [21]:
def train(model, train_loader, optimizer, criterion, scaler, run_config, run, scheduler=None, finish=True):
  """
  Trains the model and logs metrics/artifacts to WandB.
  """
  best_acc = 0

  for epoch in range(run_config['epochs']):
      batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc='Train')

      num_correct = 0
      total_loss = 0

      for i, (x, y) in enumerate(train_loader):
          optimizer.zero_grad()

          x = x.cuda()
          y = y.cuda()

          # Updated to modern PyTorch AMP syntax
          with torch.amp.autocast('cuda'):
              outputs = model(x)
              loss = criterion(outputs, y)

          num_correct += int((torch.argmax(outputs, axis=1) == y).sum())
          total_loss += float(loss)

          batch_bar.set_postfix(
              acc="{:.04f}%".format(100 * num_correct / ((i + 1) * run_config['batch_size'])),
              loss="{:.04f}".format(float(total_loss / (i + 1))),
              num_correct=num_correct,
              lr="{:.04f}".format(float(optimizer.param_groups[0]['lr'])))

          scaler.scale(loss).backward()
          scaler.step(optimizer)
          scaler.update()

          batch_bar.update()
      batch_bar.close()

      train_loss = float(total_loss / len(train_loader))
      train_acc = 100 * num_correct / (len(train_loader) * run_config['batch_size'])
      lr = float(optimizer.param_groups[0]['lr'])

      print("Epoch {}/{}: Train Acc {:.04f}%, Train Loss {:.04f}, Learning Rate {:.04f}".format(
          epoch + 1,
          run_config['epochs'],
          train_acc ,
          train_loss,
          lr
          )
      )

      # What to log
      metrics = {
          "train_loss":train_loss,
          "train_acc": train_acc,
          'lr': lr
      }

      # Log to run
      wandb.log(metrics)

      # Step the scheduler if provided
      if scheduler is not None:
          scheduler.step()

      # Updating the model version if accuracy improves
      if train_acc > best_acc:
        best_acc = train_acc

        # 1. Save the model locally
        model_path = "Model.pth"
        torch.save({
              'model_state_dict': model.state_dict(),
              'optimizer_state_dict': optimizer.state_dict()
              }, model_path)

        # 2. Create a WandB Artifact to track this model file
        model_artifact = wandb.Artifact(name=run_config['model'], type='model')
        model_artifact.add_file(model_path)

        # 3. Log the artifact to the current run
        run.log_artifact(model_artifact)

  if finish:
    wandb.finish()

## Simple Usage

You can run the training function and log the performance metrics of your choice into the WandB GUI. This simple method will allow you to monitor trends in a specific run, and compare different runs.

The cell below is where you set different values for your hyperparameters for different training runs. THIS IS IMPORTANT.

In [22]:
run_config = {
    'user_name': 'ron',
    'model': '1-2dcnn',
    'optimizer':'sgd',
    'lr': 2e-3,
    'batch_size':64,
    'epochs': 10
}

train_loader, test_loader = build_data(run_config['batch_size'], data_train, data_test)

optimizer = get_optim(run_config['optimizer'], run_config['lr'], model)

criterion = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler('cuda')

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=run_config['epochs'])

### Initializing a Run
Before calling the train function, we must start a WandB run using **`wandb.init()`**.
We pass in our project name and the `run_config` dictionary. Passing the config is a best practice because it allows WandB to track exactly which hyperparameters (learning rate, batch size, etc.) produced the resulting metrics!

In [23]:
run = wandb.init(
    project="wandb-quickstart",
    name=f"Run_2_{run_config['user_name']}_{run_config['model']}_{run_config['optimizer']}",
    config=run_config,
    #entity="wandb-starter",
    #job_type="model-training",
    )

### Training the model

This is where you actaully train your model.

In [24]:
train(model, train_loader, optimizer, criterion, scaler, run_config, run, scheduler=scheduler)

Epoch 1/10: Train Acc 37.6978%, Train Loss 1.7996, Learning Rate 0.0020


Epoch 2/10: Train Acc 38.8187%, Train Loss 1.7764, Learning Rate 0.0020


Epoch 3/10: Train Acc 39.4481%, Train Loss 1.7576, Learning Rate 0.0018


Epoch 4/10: Train Acc 39.9676%, Train Loss 1.7426, Learning Rate 0.0016


Epoch 5/10: Train Acc 40.5471%, Train Loss 1.7282, Learning Rate 0.0013


...

Epoch 7/10: Train Acc 41.0945%, Train Loss 1.7130, Learning Rate 0.0007


Epoch 8/10: Train Acc 41.4043%, Train Loss 1.7076, Learning Rate 0.0004


Epoch 9/10: Train Acc 41.6460%, Train Loss 1.7044, Learning Rate 0.0002


Epoch 10/10: Train Acc 41.4702%, Train Loss 1.7049, Learning Rate 0.0000


lr,██▇▇▆▄▃▂▂▁
train_acc,▁▃▄▅▆▇▇███
train_loss,█▆▅▄▃▂▂▁▁▁
lr,5e-05
train_acc,41.47019
train_loss,1.70486


### Resume a previous run

In [ ]:
RESUME_LOGGING = True ### Change to true to test the code for resuming an existing run

In [ ]:
if RESUME_LOGGING:
  run_id = "h0vciepb" ### Replace with run id string
  run = wandb.init(
      id     = run_id, ### Insert specific run id here if you want to resume a previous run
      resume = "must", ### You need this to resume previous runs, but comment out reinit = True when using this
      project = "wandb-quickstart", ### Project should be created in your wandb account
  )

  print(run.dir)

/content/wandb/run-20241219_162020-n8cxgzyc/files


## HyperParameter Sweeps


[Sweeps](https://docs.wandb.ai/guides/sweeps) are a way of automating hyperparameter tuning in Deep Learning Models. You set up the values that you want your sweep to try and then check the affect of changing each parameter on each value on the model.

### Setting up the Sweep Configuration
A sweep requires a configuration dictionary to tell the WandB agent what to do. You must specify:
1. **Method**: How to search the hyperparameter space (e.g., `random`, `grid`, or `bayes`).
2. **Metric**: The target metric to optimize (e.g., minimize loss).
3. **Parameters**: The hyperparameters and their possible values or distributions to search through.

In [ ]:
# Initialize the sweep and set the method (grid, random or bayesian)

sweep_config = {
    'method': 'random'
    }

In [ ]:
# What is the objective of the sweep (minimize loss, maximize accuracy)

metric = {
    'name':'loss',
    'goal':'minimize'
}
sweep_config['metric'] = metric

In [ ]:
# Hyperparameters to work with

parameters_dict = {
    'optimizer':{
        'values': ['sgd', 'adam']
    },
    'learning_rate':{
        'distribution':'uniform',
        'min':2e-4,
        'max':1e-1
    },
    'batch_size': {
        'distribution': 'q_log_uniform_values',
        'q':4,
        'min': 16,
        'max': 128
    },
    'epochs':{
        'value': 5
    }
}
sweep_config['parameters'] = parameters_dict

In [ ]:
# Initalizing the sweep

sweep_id = wandb.sweep(sweep_config, project="CIFAR-Sweep2")

Create sweep with ID: tz3x5p3x
Sweep URL: https://wandb.ai/nzafloris/CIFAR-Sweep2/sweeps/tz3x5p3x


In [ ]:
def train_sweep(config = None):
    with wandb.init(config=config) as run:
        run.name=f"Sweep_{wandb.config.learning_rate}_{wandb.config.batch_size}_{wandb.config.optimizer}"
        config = wandb.config

        train_loader, test_loader = build_data(config.batch_size, data_train, data_test)

        model = Network().to(device)

        optimizer = get_optim(config.optimizer, config.learning_rate, model)

        criterion = nn.CrossEntropyLoss()

        # Updated to modern PyTorch AMP syntax
        scaler = torch.amp.GradScaler('cuda')

        for epoch in range(config.epochs):

            model, loss = train_epoch(model, train_loader, optimizer, criterion, scaler)

            wandb.log({'loss': loss})

In [ ]:
# Running the sweep

wandb.agent(sweep_id, train_sweep, count=2)

## Artifact and Model Versioning

Artifacts are a method of managing versions for data and models. You can use the artifacts to keep and compare versions of your model while training making it easier to share data and models between team members.

### Why use Artifacts?
Instead of just saving a `.pth` file locally and losing track of which parameters created it, Artifacts link your saved model directly to the run that produced it. You can track lineage, compare versions, and easily download them later, as we will demonstrate in the following cells.

In [ ]:
run_config = {
    'model': '1-2dcnn',
    'optimizer':'adam',
    'lr': 5e-3,
    'batch_size':20,
    'epochs': 5
}

train_loader, test_loader = build_data(run_config['batch_size'], data_train, data_test)
optimizer = get_optim(run_config['optimizer'], run_config['lr'], model)
criterion = nn.CrossEntropyLoss()

# Updated to modern PyTorch AMP syntax
scaler = torch.amp.GradScaler('cuda')

# Initialize a learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=run_config['epochs'])

In [ ]:
run = wandb.init(
    project="wandb-quickstart",
    job_type="model-training",
    name=run_config['model'],
    config=run_config
    )

In [ ]:
# Call the updated train function with all explicit dependencies, including the scheduler
train(model, train_loader, optimizer, criterion, scaler, run_config, run, scheduler=scheduler, finish=False) #run should not finish for using artifact

In [ ]:
## Retreiving the model

# Getting the latest version of the artifact
artifact = run.use_artifact('{}:latest'.format(run_config['model']))
# Downloading the artifact
artifact_dir = artifact.download()
# Loading the model (Updated to match the saved 'Model.pth' filename)
model_dict = torch.load(os.path.join(artifact_dir, 'Model.pth'))

# Loading weights
model.load_state_dict(model_dict['model_state_dict'])
# Loading optimizer state
optimizer.load_state_dict(model_dict['optimizer_state_dict'])

In [ ]:
# Finishing runs
wandb.finish()